<a href="https://colab.research.google.com/github/olucasaguiar/estudos-sobre-machine-learning/blob/main/temas/natural-language-processing/NLP_lucas_nascimento_aguiar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import nltk
import pandas as pd
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

TrainSet = tuple[pd.DataFrame, pd.Series]
TestSet = tuple[pd.DataFrame, pd.Series]


def load_dataset(handle: str, path: str) -> pd.DataFrame:
  return kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    handle,
    path,
  )

def prepare_dataset(
    feature_values: pd.DataFrame | pd.Series,
    target_values: pd.DataFrame | pd.Series,
    max_features: int | None=None,
    max_df: float=1.0,
    min_df: int | float=1,
    norm: str | None='l2',
  ) -> tuple[TrainSet, TestSet]:
  # step 1: Split test and train data
  X_train, X_test, y_train, y_test = train_test_split(feature_values, target_values, test_size=0.33, random_state=42, shuffle=True)

  # step 2: Create TF-IDF Vectorizer for documents vocabulary
  vectorizer = TfidfVectorizer(
    strip_accents='ascii',
    stop_words=stopwords.words('portuguese'),
    norm=norm,
    max_df= max_df,
    min_df= min_df,
    max_features=max_features,
  )

  # step 3: Fit data into vetorizer/tokenizer and transform feature_data
  X_train = vectorizer.fit_transform(X_train)
  X_test = vectorizer.transform(X_test)

  # step 4: Return train and test datasets
  return (X_train, y_train), (X_test, y_test)

## setup
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Sobre o dataset

O UTLCorpus, proveniente do projeto Opinando, é um corpus com mais de 2 milhões de avaliações. Ele engloba resenhas de filmes coletadas do Filmow, uma popular rede social de filmes, e foi tratado especificamente como UTLC-Movies neste contexto, com todas as avaliações que possuíam classificação 0 removidas para garantir a qualidade dos dados.

In [ ]:
from re import X
dataset_handle = "fredericods/ptbr-sentiment-analysis-datasets"
dataset_file = "utlc_movies.csv"

# loading kaggle dataset
df = load_dataset(handle=dataset_handle, path=dataset_file)
print(df.sample())

# selecting columns
df = df[["review_text", "polarity", "rating"]]
print(df.head())

100%|██████████| 318M/318M [00:03<00:00, 102MB/s]

Extracting zip of utlc_movies.csv...


        original_index                                        review_text  \
707464           61663  Acho que é mais do que gosto, pode ser tbm est...   

                                    review_text_processed  \
707464  acho que e mais do que gosto, pode ser tbm est...   

                                    review_text_tokenized  polarity  rating  \
707464  ['acho', 'que', 'mais', 'do', 'que', 'gosto', ...       NaN       3   

        kfold_polarity  kfold_rating  
707464              -1             5  
                                         review_text  polarity  rating
0                         Um dos melhores desenhos!!       1.0       4
1  O filme é realmente diferente e bem lento mas ...       1.0       4
2     Hilário em alguns momentos, e muito bem feito.       1.0       4
3                         choro toda vez que vejo :(       1.0       5
4                                           Niiiice!       1.0       4


In [ ]:
df_backup = df.copy()

In [ ]:
# filtering NaN columns
import numpy as np


df = df_backup
df_nan = df[np.isnan(df["polarity"])]
df_filtered = df[~np.isnan(df["polarity"])]

print((len(df) - len(df_filtered)) == len(df_nan))

df = df_filtered

True


In [ ]:
# df = df_backup
(X_train, y_train), (X_test, y_test) = prepare_dataset(feature_values=df["review_text"], target_values=df["polarity"])

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ate', 'eramos', 'estao', 'estavamos', 'estiveramos', 'estivessemos', 'foramos', 'fossemos', 'ha', 'hao', 'houveramos', 'houverao', 'houveriamos', 'houvessemos', 'ja', 'nao', 'sao', 'sera', 'serao', 'seriamos', 'so', 'tambem', 'tera', 'terao', 'teriamos', 'tinhamos', 'tiveramos', 'tivessemos', 'voce', 'voces'] not in stop_words.
  warnings.warn(


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("| qtd. documentos | tam. vocabulario |")
print("| --------------- | ---------------- |")
print("| %15d | %16d |" % X_train.shape)

print(len(y_train[np.isnan(y_train)]))

| qtd. documentos | tam. vocabulario |
| --------------- | ---------------- |
|          796993 |           202702 |
0


In [ ]:
from sklearn.metrics import classification_report
from sklearn.neural_network import MLPClassifier

clf = MLPClassifier(
  hidden_layer_sizes=(50,),
  activation='relu',
  solver='adam',
  batch_size=200,
  learning_rate='constant',
  learning_rate_init=0.001,
  max_iter=200,
  random_state=42,
)
clf.fit(X_train, y_train)

predictions = clf.predict(X_test)
print("Classification Report:\n", classification_report(y_test, predictions))

In [ ]:
from google import genai
from google.colab import userdata
from google.genai import types


def generate_content(prompt: str, model: str='gemini-2.5-flash-lite', system_instruction: str|None=None,  temperature: float|None=None, top_p: float|None=None) -> str:
  try:
    genai_api_key=userdata.get('GOOGLE_API_KEY')
  except userdata.SecretNotFoundError as e:
    print('Secret not found. Configure secret for Gemini Models using secret variable \"GOOGLE_API_KEY\".')
    raise e

  with genai.Client(api_key=genai_api_key) as client:
    response = client.models.generate_content(
      model=model,
      contents=types.Part.from_text(text=prompt),
      config=types.GenerateContentConfig(
          system_instruction=system_instruction,
          temperature=temperature,
          top_p=top_p,
      ),
    )
    return response.text

In [ ]:
from io import StringIO
from sklearn.utils import shuffle
import pandas as pd

few_shot_prompt = """
Prepare um conjunto de dados estruturados no formato `.csv` no idioma português (pt_BR) com as dimensões:
- **index**: Valor numérico sequêncial iniciando em 1;
- **review_text**: Um texto entre 5 e 25 palavras contendo a opinião sobre um filme qualquer;
- **polarity**: A intenção do comentário, onde 0 representa negativo e 1 representa positivo.

Cada linha deve simular a avaliação de uma pessoa diferente, com suas características únicas e pessoais.
O público respondente possui faixa etária de 20 à 45 anos, e variação demográfica (genêro, contexto social, crença, etc).

**Exemplos**

Usuário: Prepare um conjunto de 5 analises positivas e 5 análises negativas
Resposta:
```csv
index,review_text,polarity
1,"Esse filme me deixou maravilhado, a história é envolvente e os atores são incríveis. Uma obra-prima!",1
2,"Adorei cada segundo! A fotografia é deslumbrante e a trilha sonora emocionou demais. Recomendo muito!",1
3,"Que experiência cinematográfica fantástica! Os efeitos visuais são de outro mundo e a trama me prendeu do início ao fim.",1
4,"Um filme que aquece o coração. A mensagem é poderosa e me fez refletir bastante sobre a vida e o amor.",1
5,"Simplesmente sensacional! Roteiro inteligente, atuações impecáveis e uma direção que me transportou para outro universo.",1
6,"Que decepção profunda! A história era confusa e o final me deixou com um gosto amargo na boca. Não gostei.",0
7,"Um desperdício de tempo. O filme foi arrastado, sem emoção e com diálogos rasos. Não recomendo de forma alguma.",0
8,"Odiei a atuação dos personagens principais. Pareciam forçados e sem carisma algum. Um filme muito fraco.",0
9,"Achei o enredo previsível e sem originalidade. Faltou criatividade e surpreendentes reviravoltas. Muito meh.",0
10,"Não curti nada neste filme. Os efeitos eram amadores, o som péssimo e a narrativa sem nexo. Paguei caro pra me frustrar.",0
```

Usuário: Prepare um conjunto de 3 analises positivas e 5 análises negativas
Resposta:
```csv
index,review_text,polarity
1,"Amei a atuação! Os personagens eram cativantes e a história me prendeu do começo ao fim. Um filme realmente inspirador.",1
2,"Que surpresa agradável! A fotografia era espetacular e a trilha sonora me tocou profundamente. Recomendo para todos!.",1
3,"Um filme que me fez rir e chorar. A mensagem sobre família e amizade é linda e muito bem contada.",1
4,"Que decepção! A trama parecia promissora, mas se perdeu em clichês e previsibilidade. Não valeu o ingresso.",0
5,"O pior filme que vi este ano. Atuações pífias e um roteiro sem pé nem cabeça. Arrependimento total.",0
6,"Não consegui me conectar com nenhum personagem. A história foi arrastada e o final me deixou frustrado e sem resposta.",0
7,"Um desastre cinematográfico. Diálogos forçados, efeitos visuais de quinta categoria e uma falta de originalidade gritante.",0
8,"Paguei caro para ver algo tão medíocre. A narrativa é confusa e não empolga em nenhum momento. Perda de tempo.",0
```

**Solicitação**
Usuário: Prepare um conjunto de {n_positive} análises positivas e {n_negative} análises negativas
Resposta:
""".strip()

def simulate_review(n_positive: int, n_negative: int) -> pd.DataFrame:
  prompt = few_shot_prompt.format(n_positive=n_positive, n_negative=n_negative)
  response = generate_content(prompt=prompt, temperature=0.3, top_p=1.0)

  if response.startswith("```csv"):
    response = response[6:]

  if response.endswith("```"):
    response = response[:-3]

  buffer = StringIO(response)
  review_df = pd.read_csv(buffer)
  review_df = shuffle(review_df, random_state=42).reset_index(drop=True)
  return review_df[['review_text', 'polarity']]

In [ ]:
llm_data = simulate_review(n_positive=50, n_negative=50)

print("| qtd. análises | qtd. positivos | qtd. negativos |")
print("| ------------- | -------------- | -------------- |")
print("| %12d | %14d | %14d |" % (len(llm_data), len(llm_data[llm_data['polarity'] == 1]), len(llm_data[llm_data['polarity'] == 0])))